# Feature Engineering & Preprocessing Pipeline

## RabTech Academy — Artificial Intelligence & Machine Learning Internship

**Objective:** Build a reusable, leak-free Scikit-Learn preprocessing pipeline for a mixed-type tabular dataset. The notebook demonstrates train/test splitting before transformations, missing-value imputation, numerical scaling, categorical one-hot encoding, `ColumnTransformer`, `Pipeline`, and correlation analysis.

> **Leakage prevention:** the train/test split is performed before fitting any preprocessing transformer. Imputers, scalers, and encoders are fitted only through the training data inside the pipeline.


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score


## 2. Load a Complex Tabular Dataset

We use the **Ames Housing** dataset from OpenML. It contains numerical and categorical predictors and a continuous house-price target, making it suitable for demonstrating mixed-type preprocessing.


In [ ]:
housing = fetch_openml(name="house_prices", as_frame=True)
df = housing.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# The OpenML house_prices dataset uses SalePrice as the target.
target = "SalePrice"
X = df.drop(columns=[target])
y = pd.to_numeric(df[target], errors="coerce")

print("Target:", target)
print("Features:", X.shape[1])
print("Missing values:", int(X.isna().sum().sum()))


## 3. Inspect Data Types and Missing Values


In [ ]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("\nTop missing-value columns:")
display(X.isna().sum().sort_values(ascending=False).head(15))


## 4. Correlation Analysis

Correlation is calculated for numerical variables only. It is used for exploratory analysis, not as a transformation fitted on the full dataset.


In [ ]:
corr = pd.concat([X[numeric_features], y.rename(target)], axis=1).corr(numeric_only=True)
target_corr = corr[target].drop(target).abs().sort_values(ascending=False)

print("Top numerical correlations with SalePrice:")
display(target_corr.head(15))


In [ ]:
plt.figure(figsize=(10, 6))
target_corr.head(15).sort_values().plot(kind="barh")
plt.title("Top Numerical Feature Correlations with SalePrice")
plt.xlabel("Absolute Pearson Correlation")
plt.tight_layout()
plt.show()


## 5. Train/Test Split — Before Preprocessing

**Important:** The split happens before fitting imputers, scalers, or encoders. This prevents information from the test set from influencing preprocessing parameters.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 6. Numerical and Categorical Pipelines

Numerical columns use median imputation followed by standard scaling. Categorical columns use most-frequent imputation followed by one-hot encoding. `handle_unknown='ignore'` makes the pipeline robust when a category appears in test/new data that was not observed during training.


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])


## 7. Build ColumnTransformer


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

preprocessor


## 8. Fit and Transform Without Leakage

The preprocessor is fitted only on `X_train`. The same fitted transformation is then applied to `X_test`.


In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


In [ ]:
feature_names = preprocessor.get_feature_names_out()
X_train_processed_df = pd.DataFrame(
    X_train_processed, columns=feature_names, index=X_train.index
)
X_test_processed_df = pd.DataFrame(
    X_test_processed, columns=feature_names, index=X_test.index
)

display(X_train_processed_df.head())


## 9. Verify Missing Values After Preprocessing


In [ ]:
print("Missing values in processed training data:", int(X_train_processed_df.isna().sum().sum()))
print("Missing values in processed test data:", int(X_test_processed_df.isna().sum().sum()))


## 10. Complete Reusable ML Pipeline

The preprocessing stage can be combined with a model so that preprocessing and prediction remain one reusable object.


In [ ]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

model_pipeline.fit(X_train, y_train)
predictions = model_pipeline.predict(X_test)

print("MAE:", mean_absolute_error(y_test, predictions))
print("R²:", r2_score(y_test, predictions))


## 11. Leakage Check

The pipeline follows a leak-free sequence:

1. Split the raw data into training and testing sets.
2. Fit numerical imputation and scaling only on training data.
3. Fit categorical imputation and one-hot encoding only on training data.
4. Transform the test set using the already-fitted training transformations.
5. Fit the model through the same pipeline.

This structure prevents test-set statistics from being used to learn preprocessing parameters.


## 12. Conclusion

A reusable Scikit-Learn preprocessing pipeline was implemented using `Pipeline` and `ColumnTransformer`. The workflow handles missing values, scales numerical variables, one-hot encodes categorical variables, performs correlation analysis, and preserves train/test separation to reduce data leakage risk.
